In [22]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "microsoft/phi-2"  # 2.7B, fully open, no gating

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [24]:
for i, layer in enumerate(model.model.layers):
    print(f"Layer {i}: {layer.__class__.__name__}")
    for name, module in layer.named_modules():
        if name:
            print(f"  {name}: {module.__class__.__name__}")

Layer 0: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 1: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 2: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 3: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj:

In [25]:
import torch.nn as nn

In [27]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=512 ):
        super().__init__()
        self.linear = linear
        self.linear.weight.requires_grad = False
        
        in_features = linear.weight.shape[1]
        out_features = linear.weight.shape[0]
        
        self.A = nn.Linear(in_features, rank, bias = False)
        self.B = nn.Linear(rank, out_features, bias = False)
        
        nn.init.normal_(self.A.weight)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.linear(x) + self.B(self.A(x))
    

In [29]:
for param in model.parameters():
    param.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,}")

Trainable: 0 / 2,779,683,840


In [28]:
for i, layer in enumerate(model.model.layers):
    print(f"Layer {i}: {layer.__class__.__name__}")
    for name, module in layer.named_modules():
        if name:
            print(f"  {name}: {module.__class__.__name__}")

Layer 0: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 1: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 2: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.dense: Linear
  mlp: PhiMLP
  mlp.activation_fn: NewGELUActivation
  mlp.fc1: Linear
  mlp.fc2: Linear
  input_layernorm: LayerNorm
  resid_dropout: Dropout
Layer 3: PhiDecoderLayer
  self_attn: PhiAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: